# 05 v2 · Training examples, version 2 (fixes the letter loop)

**Why a version 2:** after training on version 1, Tiny Aya learned the workflow, but in about half the cases it got
stuck in `submit_review`, writing the same letter sentence again and again until the reply was cut off. Three causes,
three fixes:
1. the training letter was NTSB's real letter, full of facts the model never saw → now a **short fixed letter**:
   thank-you line + NTSB's standard sentence for the grade (names the case, ends with the classification)
2. the model never had to say **what the organisation actually intends** → now `reasons` is written **first**, as
   3 steps: what NTSB asked, what the organisation says, and what that means (done / doing / alternative /
   not doing / no longer applies). Only then the grade. Think first, then decide
3. it never saw how to recover from a broken reply → about 1 in 9 runs is a **recovery run**: a cut-off
   `submit_review` (not learned, only shown), then "Your reply had no tool call", then a correct short `submit_review`

Phase 4, part 1. In `04` the untrained Tiny Aya:
- copied the last grade ("Open – Acceptable" almost every time)
- never read NTSB's own latest letter
- wrote an "NTSB letter" that was just a copy of the organisation's letter, without the case ID
- after `check_rules` said FAIL, sent **the same draft again** instead of fixing it
- often never reached `request_approval`

Here I build **worked runs**: whole conversations where the job is done **right**, using **real tool results**.
Training (next notebook) shows Tiny Aya hundreds of these, so it learns the habit.

**Every worked run, step by step (one tool call per step):**
1. `get_recommendation`
2. `read_letter` the letter to review (page 2 too, if it has one)
3. `read_letter` NTSB's latest letter (if there is one)
4. `find_precedents` (3 similar past cases) and `get_peers`
5. `verify_quote` on a real sentence from the letter
6. `submit_review` with **NTSB's real grade**, reasons from **NTSB's real answer letter**, the verified quote, and
   **NTSB's real letter** as the draft letter (it names the case and ends with the classification)
7. `check_rules` → READY
8. `request_approval`

About 1 in 7 runs is a **repair run**: the first draft forgets the case ID, `check_rules` says FAIL, and the next
draft fixes it. That's exactly the mistake from `04`.

**Same format as `04`, exactly:** same system prompt, same "Review case …", same JSON-list tool calls, same
"Tool result for …" + desk line. If the format differed, the trained model would write something `04` can't read.

**Runtime: CPU is enough** (no model, only the tokenizer for counting tokens). Runtime → Change runtime type → CPU.

## Setup
Same as `04`: AgentTune (for the same tool-calling text), our package from GitHub, Drive, Hugging Face login.
After cell 1: **Runtime → Restart session**, then run from cell 2. Don't run cell 1 again.

In [ ]:
!pip install -q git+https://github.com/Lexsi-Labs/AgentTune.git rank_bm25

In [ ]:
import os, sys
if os.path.exists("/content/ntsb-safety-review-agent"):
    !cd /content/ntsb-safety-review-agent && git pull -q
else:
    !git clone -q https://github.com/niharnandala/ntsb-safety-review-agent.git /content/ntsb-safety-review-agent
sys.path.append("/content/ntsb-safety-review-agent")

from google.colab import drive, userdata
drive.mount("/content/drive")
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

DATA = "/content/drive/MyDrive/ntsb_agent/data"
RESULTS = "/content/drive/MyDrive/ntsb_agent/results"
TRAINING = "/content/drive/MyDrive/ntsb_agent/training"
os.makedirs(TRAINING, exist_ok=True)

In [ ]:
import json, random, re
from ntsb_agent.data import load_all
from ntsb_agent import desk
from ntsb_agent.desk import open_case, run_tool
from ntsb_agent import tools
from ntsb_agent.settings import GRADES, PAGE_SIZE, LETTER_LIMIT
from ntsb_agent.grades import last_ntsb_grade
from ntsb_agent.tools.precedents import tokens, MAIL_HEADER

train, val, test = load_all(DATA)
tools.setup(train, val, test)
print(len(desk.TOOLS), "tools:", ", ".join(desk.TOOLS))

## The tokenizer (to count the desk exactly like `04`)
Only the tokenizer, not the model. Every "Tool result" message ends with the desk line
`[desk: 2,061 of 7,492 tokens used]`, and the numbers must be the real ones, the same `04` would show.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global")

CONTEXT = 8192
REPLY_ROOM = 700
DESK_LIMIT = CONTEXT - REPLY_ROOM


def count_tokens(messages):
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_dict=True)["input_ids"]
    return len(ids)


def desk_line(used):
    line = f"[desk: {used:,} of {DESK_LIMIT:,} tokens used]"
    if used > 0.8 * DESK_LIMIT:
        line = line + " Your desk is almost full: submit your review and ask for approval now."
    return line

## The system prompt: copied word for word from `04`
Then I check it against a real conversation `04` saved, so I **know** it's identical.

In [ ]:
from transformers.utils import get_json_schema
from agenttune.agentic.rollout_engines.rollout_factory import DEFAULT_TOOLS_FALLBACK_PROMPT
from agenttune.agentic.rollout_engines.tool_call_parse import extract_tool_calls_from_text

TOOL_SCHEMAS = []
for name in desk.TOOLS:
    TOOL_SCHEMAS.append(get_json_schema(desk.TOOLS[name]))

grade_list = ""
for label in GRADES:
    grade_list = grade_list + "- " + label + "\n"

TASK = (
    "You assist an NTSB safety analyst. You get one case. Review the newest letter from the organisation, "
    "propose the grade NTSB should give it, and send your review for human approval.\n\n"
    "Steps:\n"
    "1. get_recommendation\n"
    "2. read_letter: the letter to review, and NTSB's latest letter\n"
    "3. find_precedents and get_peers: how NTSB graded similar replies\n"
    "4. submit_review: grade, confidence (0 to 1), reasons, an exact quote of 4+ words from the letter you review, "
    "and a short NTSB letter that names the case and ends with the classification\n"
    "5. check_rules: if anything says FAIL, fix it and submit_review again\n"
    "6. request_approval\n\n"
    "Rules: never invent quotes. Use only these grades:\n" + grade_list +
    "You cannot approve your own review. Your desk is small: read only what you need.\n\n"
)

tool_rules = DEFAULT_TOOLS_FALLBACK_PROMPT.format(tools=json.dumps(TOOL_SCHEMAS))

# AgentTune's text ends with "...use it to answer the user's original question as plain text."
# Our job is the opposite: keep calling tools until the review is sent. So we swap that last sentence.
AGENTTUNE_END = "not a new message from the user: use it to answer the user's original question as plain text."
OUR_END = ("not a new message from the user. Keep working with tool calls until request_approval says "
           "your review was sent. Never answer in plain text.")
if AGENTTUNE_END not in tool_rules:
    raise ValueError("AgentTune's prompt text changed: check the last sentence by hand")
tool_rules = tool_rules.replace(AGENTTUNE_END, OUR_END)

SYSTEM_PROMPT = TASK + tool_rules
print("system prompt:", len(tok(SYSTEM_PROMPT)["input_ids"]), "tokens")

In [ ]:
CONV_FOLDER = f"{RESULTS}/before_untrained_conversations"
checked = False
if os.path.exists(CONV_FOLDER):
    for name in sorted(os.listdir(CONV_FOLDER)):
        with open(f"{CONV_FOLDER}/{name}") as f:
            saved = json.load(f)
        if len(saved) > 0:
            if saved[0]["content"] == SYSTEM_PROMPT:
                print("✅ identical to the system prompt in", name)
            else:
                print("❌ DIFFERENT from the system prompt in", name, "→ stop and check")
            checked = True
            break
if not checked:
    print("no saved 04 conversation found yet, check skipped")

## Small helpers
- `latest_ntsb_letter`: NTSB's newest letter in the case (the same one `get_recommendation` names)
- `pick_quote`: the sentence of the reply that shares the most words with NTSB's real answer (6 to 40 words),
  so the evidence is a sentence NTSB actually reacted to
- `make_reasons`: the model's **thinking, written before the grade**, in 3 short steps:
  *NTSB asked: …* → *The organisation says: "…"* → *So they have done it / are doing it / are doing something
  different / are not doing it / say it no longer applies* → *Does it match the request? yes / partly / no* →
  NTSB's last grade, what similar past cases and other organisations got (only what the tools showed).
  The grade comes right after.
- `make_letter`: a **short fixed letter**: thank-you line + NTSB's standard sentence for the grade
- `confidence_for`: 0.85 if NTSB kept the last grade, 0.7 if it changed, 0.75 if there was no grade before

In [ ]:
def latest_ntsb_letter(case):
    latest = None
    for letter in case["history"]:
        if letter["from"] == "NTSB":
            latest = letter["letter_id"]
    return latest


def sentences(text):
    text = " ".join(text.split())
    return re.split(r"(?<=[.!?])\s+", text)


def pick_quote(reply_text, ntsb_text):
    ntsb_words = set(tokens(ntsb_text))
    clean = MAIL_HEADER.sub(" ", reply_text)
    candidates = []
    number = 0
    for sentence in sentences(clean):
        number = number + 1
        words = sentence.split()
        if len(words) < 6 or len(words) > 40:
            continue
        score = 0
        for word in set(tokens(sentence)):
            if word in ntsb_words:
                score = score + 1
        candidates.append([-score, number, sentence])     # most shared words first, then reading order
    candidates.sort()
    found = []
    for item in candidates:
        found.append(item[2])
    return found


INTENT = {
    "Open – Acceptable": "are doing what NTSB asked, but are not finished yet",
    "Open – Acceptable Alternate": "are doing something different from what NTSB asked that still meets its intent, but are not finished yet",
    "Open – Unacceptable": "are not doing what NTSB asked, or what they plan does not meet the request",
    "Closed – Acceptable": "have done what NTSB asked",
    "Closed – Acceptable Alternate": "have done something different from what NTSB asked that meets its intent",
    "Closed – Unacceptable": "will not do what NTSB asked, and there is no point asking again",
    "Closed – Other": "say the recommendation no longer applies (it was replaced, or the situation changed)",
}


def first_words(text, n):
    words = text.split()
    if len(words) <= n:
        return " ".join(words)
    return " ".join(words[:n]) + " ..."


MATCHES = {
    "Open – Acceptable": "yes, the same thing NTSB asked for",
    "Open – Acceptable Alternate": "partly, a different way to reach the same goal",
    "Open – Unacceptable": "no",
    "Closed – Acceptable": "yes, the same thing NTSB asked for",
    "Closed – Acceptable Alternate": "partly, a different way to reach the same goal",
    "Closed – Unacceptable": "no",
    "Closed – Other": "the request no longer applies",
}


def count_grades(tool_text):
    # "NTSB GRADE: Closed – Acceptable" lines in the find_precedents result -> {grade: count}
    counts = {}
    for line in tool_text.split("\n"):
        if "NTSB GRADE:" in line:
            g = line.split("NTSB GRADE:")[1].strip()
            if g not in counts:
                counts[g] = 0
            counts[g] = counts[g] + 1
    return counts


def grades_text(counts):
    parts = []
    for g in counts:
        parts.append(f"{counts[g]} × {g}")
    return ", ".join(parts)


def make_reasons(case, grade, quote, last_grade, precedent_text="", peer_text=""):
    # think first, in 3 short steps, then the grade follows:
    # 1. what NTSB asked (in short)  2. what the organisation says  3. what that means
    asked = case["recommendation"]
    colon = asked.find(":")
    if colon != -1 and colon < 150:
        asked = asked[colon + 1:]
    asked = first_words(asked, 25)
    said = first_words(quote, 20)
    text = f"NTSB asked: {asked} The organisation says: \"{said}\" So they {INTENT[grade]}."
    text = text + f" Does it match the request? {MATCHES[grade]}."
    if last_grade is not None:
        text = text + f" NTSB's last grade was {last_grade}."
    # what NTSB did before with similar replies (only what the tools showed on the desk)
    counts = count_grades(precedent_text)
    if len(counts) > 0:
        text = text + f" Similar past cases: {grades_text(counts)}."
    if "Latest grades before this reply:" in peer_text:
        peers_line = peer_text.split("Latest grades before this reply:")[1].split("\n")[0].strip()
        text = text + f" Other organisations: {peers_line}."
    return text


GRADE_SENTENCE = {
    "Open – Acceptable": "We are encouraged by the actions you describe. Pending their completion, Safety Recommendation {case} is classified OPEN—ACCEPTABLE RESPONSE.",
    "Open – Acceptable Alternate": "The alternate approach you describe would meet the intent of the recommendation. Pending its completion, Safety Recommendation {case} is classified OPEN—ACCEPTABLE ALTERNATE RESPONSE.",
    "Open – Unacceptable": "The actions you describe do not meet the intent of the recommendation. Safety Recommendation {case} is classified OPEN—UNACCEPTABLE RESPONSE.",
    "Closed – Acceptable": "These actions satisfy the intent of the recommendation. Safety Recommendation {case} is classified CLOSED—ACCEPTABLE ACTION.",
    "Closed – Acceptable Alternate": "The alternate actions you took satisfy the intent of the recommendation. Safety Recommendation {case} is classified CLOSED—ACCEPTABLE ALTERNATE ACTION.",
    "Closed – Unacceptable": "Because the recommended action will not be taken, Safety Recommendation {case} is classified CLOSED—UNACCEPTABLE ACTION.",
    "Closed – Other": "Because the recommendation no longer applies as written, Safety Recommendation {case} is classified CLOSED—NO LONGER APPLICABLE.",
}


def make_letter(ntsb_text, case_id, grade):
    # short and fixed: only what the model can know (the case ID and its grade), so there is nothing to invent
    opening = f"Thank you for your letter regarding Safety Recommendation {case_id}. "
    return opening + GRADE_SENTENCE[grade].replace("{case}", case_id)


def confidence_for(grade, last_grade):
    if last_grade is None:
        return 0.75
    if grade == last_grade:
        return 0.85
    return 0.7

## Building one worked run
`add_call` does what `04`'s loop does, but here **I** write the assistant turn instead of Tiny Aya:
1. the assistant message: one JSON list with one call
2. `run_tool` runs the real tool (and writes the audit log)
3. the user message: `Tool result for …` + the desk line, counted **after** the assistant message, like `04`

If anything goes wrong (letter too long, no good quote, rules not READY, desk over the limit, a reply longer
than 600 tokens), the case is
skipped and I count why.

In [ ]:
def add_call(messages, name, params):
    messages.append({"role": "assistant",
                     "content": json.dumps([{"tool_name": name, "parameters": params}], ensure_ascii=False)})
    result = run_tool(name, params)
    feedback = f"Tool result for {name}: {result}" + "\n" + desk_line(count_tokens(messages))
    messages.append({"role": "user", "content": feedback})
    return result


NO_CALL = "Your reply had no tool call. Reply with only a JSON list of tool calls."   # the exact message from 04


def build_example(e, repair, recover=False):
    open_case(e)
    case = desk.current.case
    case_id = case["case_id"]
    grade = e["answer"]["label"]
    ntsb_text = e["answer"]["ntsb_letter"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Review case {case_id}."},
    ]

    # 1. open the case
    add_call(messages, "get_recommendation", {})

    # 2. the letter to review (max 2 pages)
    reply_id = case["to_evaluate"]
    reply = desk.current.letter(reply_id)
    pages = len(reply["text"]) // PAGE_SIZE
    if len(reply["text"]) % PAGE_SIZE != 0:
        pages = pages + 1
    if pages > 2:
        return None, "letter longer than 2 pages"
    add_call(messages, "read_letter", {"letter_id": reply_id})
    if pages == 2:
        add_call(messages, "read_letter", {"letter_id": reply_id, "page": 2})

    # 3. NTSB's latest letter (page 1)
    ntsb_id = latest_ntsb_letter(case)
    if ntsb_id is not None:
        add_call(messages, "read_letter", {"letter_id": ntsb_id})

    # 4. the evidence quote
    quote = None
    for candidate in pick_quote(reply["text"], ntsb_text):
        check = desk.TOOLS["verify_quote"](reply_id, candidate)      # silent test first (not in the chat, not in the log)
        if check.startswith("VERIFIED"):
            quote = candidate
            break
    if quote is None:
        return None, "no usable quote"

    # 5. precedents + peers
    query_words = quote.split()
    query = " ".join(query_words[:12])
    precedent_text = add_call(messages, "find_precedents", {"query": query, "k": 3})
    peer_text = add_call(messages, "get_peers", {})

    # 6. verify the quote (in the chat this time)
    add_call(messages, "verify_quote", {"letter_id": reply_id, "quote": quote})

    # 7. the review
    last_grade, from_letter = last_ntsb_grade(case["history"], case_id)
    draft = {
        "reasons": make_reasons(case, grade, quote, last_grade, precedent_text, peer_text),   # written first: think, then decide
        "grade": grade,
        "confidence": confidence_for(grade, last_grade),
        "quote_letter_id": reply_id,
        "quote": quote,
        "letter": make_letter(ntsb_text, case_id, grade),
    }

    if repair:
        bad = dict(draft)
        bad["letter"] = draft["letter"].replace(case_id, "this recommendation")
        add_call(messages, "submit_review", bad)
        result = add_call(messages, "check_rules", {})
        if "FAIL: letter must name the case ID" not in result:
            return None, "repair run did not fail as planned"

    skip_turns = []
    if recover:
        # a cut-off submit_review (shown, NOT learned), then 04's message, then the correct short one
        full_call = json.dumps([{"tool_name": "submit_review", "parameters": draft}], ensure_ascii=False)
        broken = full_call[: len(full_call) * 2 // 3]
        skip_turns.append(len(messages))
        messages.append({"role": "assistant", "content": broken})
        messages.append({"role": "user", "content": NO_CALL})

    add_call(messages, "submit_review", draft)
    result = add_call(messages, "check_rules", {})
    if "READY for approval" not in result:
        return None, "rules not READY"

    result = add_call(messages, "request_approval", {})
    if not result.startswith("Sent for approval"):
        return None, "approval refused"

    # the run stops here in 04 too: the model never answers this last tool result
    messages.pop()

    used = count_tokens(messages)
    if used > DESK_LIMIT:
        return None, "desk over the limit"

    # every reply Tiny Aya will learn to write must fit in its 700-token reply room (with some space left)
    for m in messages:
        if m["role"] == "assistant":
            if len(tok(m["content"])["input_ids"]) > REPLY_ROOM - 100:
                return None, "a reply is too long for the reply room"
    return (messages, skip_turns), "ok"

## Piece 1: one worked run, printed in full
I look at it before making hundreds. This is exactly what Tiny Aya will learn from.

In [ ]:
sample = None
for e in train:
    if e["answer"]["label"] == "Closed – Acceptable" and len(e["history"]) >= 3:
        sample = e
        break

built, status = build_example(sample, repair=False, recover=True)
print("status:", status)
if built is not None:
    messages, skip_turns = built
    print("turns shown but not learned:", skip_turns)
    for m in messages[1:]:
        print(f"\n===== {m['role']} =====")
        print(m["content"][:1500])
    print("\ntokens:", count_tokens(messages))

## Piece 2: about 620 worked runs from the training cases
How many per grade (the real NTSB mix is very uneven: 9,048 Open – Acceptable but only 610 Open – Acceptable
Alternate). I keep the two common grades bigger, but give every grade enough examples to learn it:

| grade | runs |
|---|---|
| Open – Acceptable | 130 |
| Closed – Acceptable | 130 |
| Open – Unacceptable | 90 |
| Closed – Unacceptable | 80 |
| Closed – Acceptable Alternate | 70 |
| Closed – Other | 60 |
| Open – Acceptable Alternate | 60 |

Every 7th accepted run is a repair run, and about every 9th is a recovery run. Fixed seed (2026), so the same runs come out every time.
Takes a few minutes on CPU.

In [ ]:
TARGET = {
    "Open – Acceptable": 130,
    "Closed – Acceptable": 130,
    "Open – Unacceptable": 90,
    "Closed – Unacceptable": 80,
    "Closed – Acceptable Alternate": 70,
    "Closed – Other": 60,
    "Open – Acceptable Alternate": 60,
}


def build_many(cases, target, seed, every_repair):
    shuffled = list(cases)
    random.seed(seed)
    random.shuffle(shuffled)

    made = {}
    for grade in target:
        made[grade] = 0
    skipped = {}
    rows = []
    repairs = 0
    recovers = 0

    for e in shuffled:
        grade = e["answer"]["label"]
        if grade not in target:
            continue
        if made[grade] >= target[grade]:
            continue

        repair = False
        recover = False
        if (len(rows) + 1) % every_repair == 0:
            repair = True
        elif (len(rows) + 1) % 9 == 0:
            recover = True

        built, status = build_example(e, repair, recover)
        if built is None:
            if status not in skipped:
                skipped[status] = 0
            skipped[status] = skipped[status] + 1
            continue
        messages, skip_turns = built

        rows.append({
            "episode_id": e["episode_id"],
            "grade": grade,
            "repair": repair,
            "recover": recover,
            "skip_turns": skip_turns,
            "tokens": count_tokens(messages),
            "messages": messages,
        })
        made[grade] = made[grade] + 1
        if repair:
            repairs = repairs + 1
        if recover:
            recovers = recovers + 1

        finished = True
        for g in target:
            if made[g] < target[g]:
                finished = False
        if finished:
            break

    print("made per grade:", made)
    print("repair runs:", repairs, "| recovery runs:", recovers, "| skipped:", skipped)
    return rows


train_rows = build_many(train, TARGET, 2026, 7)
print(len(train_rows), "training runs")

## Piece 3: a small check set from `val` (not the exam)
About 60 runs from the validation cases, with the natural grade mix. Training uses them only to watch the loss
on cases it doesn't learn from. The exam cases (`test`) are never touched here.

In [ ]:
VAL_TARGET = {}
for grade in GRADES:
    VAL_TARGET[grade] = 0
for e in val:
    VAL_TARGET[e["answer"]["label"]] = VAL_TARGET[e["answer"]["label"]] + 1
for grade in VAL_TARGET:
    VAL_TARGET[grade] = max(2, round(VAL_TARGET[grade] * 60 / len(val)))

val_rows = build_many(val, VAL_TARGET, 2026, 7)
print(len(val_rows), "check runs")

## Piece 4: numbers + save to Drive

In [ ]:
total = 0
biggest = 0
steps = 0
for row in train_rows:
    total = total + row["tokens"]
    if row["tokens"] > biggest:
        biggest = row["tokens"]
    for m in row["messages"]:
        if m["role"] == "assistant":
            steps = steps + 1
print(f"average {total / len(train_rows):,.0f} tokens per run · biggest {biggest:,} (limit {DESK_LIMIT:,})")
print(f"average {steps / len(train_rows):.1f} tool calls per run")


def save_rows(rows, path):
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("saved", len(rows), "→", path)


save_rows(train_rows, f"{TRAINING}/train_examples_v2.jsonl")
save_rows(val_rows, f"{TRAINING}/val_examples_v2.jsonl")

## Piece 5: upload to Hugging Face (private for now)
The training job runs on a Hugging Face GPU, so it reads the examples from here.
Private while we work; the hackathon needs it **public** at the end (NTSB data is public US government data).

In [ ]:
from datasets import Dataset, DatasetDict

HF_DATASET = "niharnandala/ntsb-review-trajectories-v2"
dataset = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
})
dataset.push_to_hub(HF_DATASET, private=True)
print("uploaded →", f"https://huggingface.co/datasets/{HF_DATASET}")

## Where I landed
_(fill in after the run)_
- training runs: ___ (repair runs: ___) · check runs: ___
- average tokens per run: ___ · biggest: ___
- skipped and why: ___

**Next:** `06_train.py`: LoRA fine-tuning of Tiny Aya on these runs (AgentTune's LoRA settings + TRL SFT,
loss only on Tiny Aya's own turns), on a Hugging Face GPU.